In [ ]:
For Baseline model

In [ ]:
import os
import math
import json
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# =========================================================
# 1. CONFIG
# =========================================================
MODEL_MODE = "BASELINE"   # "BASELINE" or "FUSION"
DATASET_PATH = "P1"       # change if needed
SPLIT_NAME = "P1S1"       # "P1S1" or "P1S2"
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-3
NUM_WORKERS = 0           # use 0 on mac/notebooks if worker issues happen
SEED = 42
IMG_W, IMG_H = 640, 480

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("====================================")
print("Starting MMVR Pose Training")
print(f"Device      : {DEVICE}")
print(f"Model Mode  : {MODEL_MODE}")
print(f"Dataset Path: {DATASET_PATH}")
print(f"Split       : {SPLIT_NAME}")
print("====================================")

# Camera / radar params from README
R_ext = np.array([
    [0.997214252753733, -0.0607987476962386, -0.0432116463859367],
    [-0.0630518006224704, -0.996610215186671, -0.0528445779985113],
    [0.0398522840384131, -0.0554219384733665, 0.997667381541953]
], dtype=np.float32)

T_ext = np.array([
    [-0.0648588235850897],
    [ 0.302966783204636 ],
    [ 0.0914051241089986]
], dtype=np.float32)

# =========================================================
# 2. REPRODUCIBILITY
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# =========================================================
# 3. HELPERS
# =========================================================
SKELETON_CONNECTIONS = np.array([
    [13, 15], [11, 13], [14, 16], [12, 14], [11, 12],
    [5, 11], [6, 12], [5, 6], [5, 7], [6, 8], [7, 9],
    [8, 10], [1, 2], [0, 1], [0, 2], [1, 3], [2, 4],
    [3, 5], [4, 6]
], dtype=np.int32)

def normalize_heatmap(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    mean = x.mean()
    std = x.std()
    return (x - mean) / (std + 1e-6)

def get_relative_segment_path(root_dir: str, abs_dir: str) -> str:
    rel = os.path.relpath(abs_dir, root_dir)
    return rel.replace("\\", "/")

def flatten_split_entries(split_entries):
    """
    Converts split structure into a set of normalized relative folder paths.
    Usually entries are like:
      ['d1s1/000', 'd1s1/001', ...]
    """
    out = set()

    if isinstance(split_entries, np.ndarray):
        split_entries = split_entries.tolist()

    for item in split_entries:
        if isinstance(item, bytes):
            item = item.decode("utf-8")
        elif not isinstance(item, str):
            item = str(item)

        item = item.replace("\\", "/").strip().strip("/")
        out.add(item)

    return out

def masked_mse_loss(pred, gt, valid_mask):
    """
    pred: (B, 17, 2)
    gt:   (B, 17, 2)
    valid_mask: (B, 17) bool
    """
    mask = valid_mask.unsqueeze(-1).float()
    sq = (pred - gt) ** 2
    loss = (sq * mask).sum() / (mask.sum() + 1e-8)
    return loss

def pck_counts(pred, gt, valid_mask, threshold=0.10):
    """
    threshold is in normalized image coordinates.
    Returns correct_count, total_valid
    """
    distances = torch.norm(pred - gt, dim=2)   # (B,17)
    correct = ((distances < threshold) & valid_mask).sum().item()
    total = valid_mask.sum().item()
    return correct, total

def compute_pck_percent(correct, total):
    if total == 0:
        return 0.0
    return 100.0 * correct / total

def oks_per_instance(pred_xy, gt_xy, valid_mask, bbox_area, sigmas=None):
    """
    pred_xy, gt_xy: (17,2) in pixel coordinates
    valid_mask: (17,) bool
    bbox_area: scalar
    """
    if sigmas is None:
        # COCO-style approximate sigmas for 17 keypoints
        sigmas = np.array([
            0.026, 0.025, 0.025, 0.035, 0.035,
            0.079, 0.079, 0.072, 0.072, 0.062,
            0.062, 0.107, 0.107, 0.087, 0.087,
            0.089, 0.089
        ], dtype=np.float32)

    valid = valid_mask.astype(bool)
    if valid.sum() == 0:
        return 0.0

    d2 = np.sum((pred_xy - gt_xy) ** 2, axis=1)  # (17,)
    denom = 2 * (sigmas ** 2) * (bbox_area + 1e-6)
    oks = np.exp(-d2 / (denom + 1e-9))
    return float(np.mean(oks[valid]))

def estimate_depth_from_radar_boxes(bbox_hori):
    """
    Simple non-learned depth estimate from radar horizontal box.
    bbox_hori shape: (n,4) [x_min, y_min, x_max, y_max]
    x is depth axis in radar coordinate.
    For P1, usually n=1.
    """
    if bbox_hori is None or len(bbox_hori) == 0:
        return None
    depth_center = 0.5 * (bbox_hori[0, 0] + bbox_hori[0, 2])
    return float(depth_center)

# =========================================================
# 4. DATASET
# =========================================================
class RadarPoseDataset(Dataset):
    def __init__(self, root_dir, view="both", allowed_segments=None):
        self.root_dir = root_dir
        self.view = view
        self.allowed_segments = allowed_segments
        self.samples = []

        self._build_index()

    def _build_index(self):
        for root, _, files in os.walk(self.root_dir):
            rel_segment = get_relative_segment_path(self.root_dir, root)

            # Only keep exact segment folders if split is provided
            if self.allowed_segments is not None:
                if rel_segment not in self.allowed_segments:
                    continue

            radar_files = [f for f in files if f.endswith("_radar.npz")]
            for f in sorted(radar_files):
                base = f.replace("_radar.npz", "")
                radar_path = os.path.join(root, f)
                pose_path = os.path.join(root, base + "_pose.npz")
                bbox_path = os.path.join(root, base + "_bbox.npz")
                meta_path = os.path.join(root, base + "_meta.npz")

                if os.path.exists(pose_path) and os.path.exists(bbox_path):
                    self.samples.append({
                        "radar_path": radar_path,
                        "pose_path": pose_path,
                        "bbox_path": bbox_path,
                        "meta_path": meta_path if os.path.exists(meta_path) else None,
                        "segment": rel_segment,
                        "frame_id": base
                    })

        self.samples.sort(key=lambda x: (x["segment"], x["frame_id"]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        radar_data = np.load(sample["radar_path"])
        hori = normalize_heatmap(radar_data["hm_hori"])
        vert = normalize_heatmap(radar_data["hm_vert"])

        if self.view == "both":
            radar_input = np.stack([hori, vert], axis=0)     # (2,256,128)
        elif self.view == "hori":
            radar_input = np.expand_dims(hori, axis=0)       # (1,256,128)
        else:
            raise ValueError("view must be 'both' or 'hori'")

        pose_data = np.load(sample["pose_path"])
        kp_all = pose_data["kp"]   # (n,17,3)

        # P1 is single subject, so we take index 0
        kp = kp_all[0].astype(np.float32)  # (17,3)
        kp_xy = kp[:, :2].copy()
        kp_score = kp[:, 2].copy()

        # visibility / valid mask
        valid = (kp_score > 0) & ~((kp[:, 0] == 0) & (kp[:, 1] == 0))

        # normalize to [0,1]
        kp_norm = kp_xy.copy()
        kp_norm[:, 0] /= IMG_W
        kp_norm[:, 1] /= IMG_H

        bbox_data = np.load(sample["bbox_path"])
        bbox_i = bbox_data["bbox_i"]       # (n,5)
        bbox_hori = bbox_data["bbox_hori"] # (n,4)

        # area for OKS from image-plane bbox
        if len(bbox_i) > 0:
            x1, y1, x2, y2 = bbox_i[0, :4]
            bbox_area = max((x2 - x1) * (y2 - y1), 1.0)
        else:
            bbox_area = 1.0

        # simple depth proxy for later analysis
        depth_target = estimate_depth_from_radar_boxes(bbox_hori)
        if depth_target is None:
            depth_target = 0.0

        global_frame_id = -1
        if sample["meta_path"] is not None:
            meta_data = np.load(sample["meta_path"])
            if "global_frame_id" in meta_data:
                global_frame_id = int(meta_data["global_frame_id"])

        return {
            "radar": torch.tensor(radar_input, dtype=torch.float32),
            "keypoints": torch.tensor(kp_norm, dtype=torch.float32),
            "valid": torch.tensor(valid, dtype=torch.bool),
            "bbox_area": torch.tensor(bbox_area, dtype=torch.float32),
            "depth_target": torch.tensor(depth_target, dtype=torch.float32),
            "segment": sample["segment"],
            "frame_id": sample["frame_id"],
            "global_frame_id": global_frame_id
        }

# =========================================================
# 5. MODEL
# =========================================================
class RadarPoseModel(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),   # 256x128 -> 128x64

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),   # 128x64 -> 64x32

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),   # 64x32 -> 32x16

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),   # 32x16 -> 16x8

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.pose_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 17 * 2)
        )

    def forward(self, radar):
        feat = self.features(radar)
        out = self.pose_head(feat)
        out = out.view(-1, 17, 2)

        # constrain to normalized image plane
        out = torch.sigmoid(out)
        return out

# =========================================================
# 6. SPLIT LOADING
# =========================================================
def load_mmvr_split(dataset_path, split_name="P1S1"):
    split_file = os.path.join(dataset_path, "data_split.npz")
    if not os.path.exists(split_file):
        raise FileNotFoundError(f"Missing split file: {split_file}")

    split_data = np.load(split_file, allow_pickle=True)
    data_split_dict = split_data["data_split_dict"].item()

    if split_name not in data_split_dict:
        raise ValueError(f"Split {split_name} not found in data_split.npz")

    split_info = data_split_dict[split_name]

    train_segments = flatten_split_entries(split_info["train"])
    val_segments = flatten_split_entries(split_info["val"])
    test_segments = flatten_split_entries(split_info["test"])

    return train_segments, val_segments, test_segments

view_setting = "both" if MODEL_MODE == "FUSION" else "hori"
channels = 2 if MODEL_MODE == "FUSION" else 1

train_segments, val_segments, test_segments = load_mmvr_split(DATASET_PATH, SPLIT_NAME)

train_dataset = RadarPoseDataset(DATASET_PATH, view=view_setting, allowed_segments=train_segments)
val_dataset   = RadarPoseDataset(DATASET_PATH, view=view_setting, allowed_segments=val_segments)
test_dataset  = RadarPoseDataset(DATASET_PATH, view=view_setting, allowed_segments=test_segments)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples  : {len(val_dataset)}")
print(f"Test samples : {len(test_dataset)}")

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=False
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False
)

# =========================================================
# 7. TRAIN / EVAL FUNCTIONS
# =========================================================
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_valid = 0

    oks_scores = []

    for batch in loader:
        radar = batch["radar"].to(DEVICE)
        gt = batch["keypoints"].to(DEVICE)
        valid = batch["valid"].to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            pred = model(radar)
            loss = masked_mse_loss(pred, gt, valid)

            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item()

        c, t = pck_counts(pred.detach(), gt.detach(), valid.detach(), threshold=0.10)
        total_correct += c
        total_valid += t

        # OKS on CPU
        pred_np = pred.detach().cpu().numpy()
        gt_np = gt.detach().cpu().numpy()
        valid_np = valid.detach().cpu().numpy()
        bbox_area_np = batch["bbox_area"].cpu().numpy()

        for i in range(len(pred_np)):
            pred_px = pred_np[i] * np.array([IMG_W, IMG_H], dtype=np.float32)
            gt_px = gt_np[i] * np.array([IMG_W, IMG_H], dtype=np.float32)
            oks = oks_per_instance(pred_px, gt_px, valid_np[i], float(bbox_area_np[i]))
            oks_scores.append(oks)

    avg_loss = total_loss / max(len(loader), 1)
    avg_pck = compute_pck_percent(total_correct, total_valid)
    avg_oks = float(np.mean(oks_scores)) if oks_scores else 0.0

    return avg_loss, avg_pck, avg_oks

# =========================================================
# 8. TRAINING
# =========================================================
model = RadarPoseModel(in_channels=channels).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {
    "train_loss": [],
    "train_pck": [],
    "train_oks": [],
    "val_loss": [],
    "val_pck": [],
    "val_oks": []
}

best_val_pck = -1.0
best_model_path = f"best_model_{MODEL_MODE.lower()}_{SPLIT_NAME.lower()}.pth"

print("\nStarting training...\n")
for epoch in range(EPOCHS):
    train_loss, train_pck, train_oks = run_epoch(model, train_loader, optimizer=optimizer)
    val_loss, val_pck, val_oks = run_epoch(model, val_loader, optimizer=None)

    history["train_loss"].append(train_loss)
    history["train_pck"].append(train_pck)
    history["train_oks"].append(train_oks)
    history["val_loss"].append(val_loss)
    history["val_pck"].append(val_pck)
    history["val_oks"].append(val_oks)

    if val_pck > best_val_pck:
        best_val_pck = val_pck
        torch.save(model.state_dict(), best_model_path)

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f}, PCK: {train_pck:.2f}%, OKS: {train_oks:.4f} | "
        f"Val Loss: {val_loss:.4f}, PCK: {val_pck:.2f}%, OKS: {val_oks:.4f}"
    )

print(f"\nBest model saved to: {best_model_path}")

# Save history
history_path = f"history_{MODEL_MODE.lower()}_{SPLIT_NAME.lower()}.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"Training history saved to: {history_path}")

# =========================================================
# 9. TEST EVALUATION
# =========================================================
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
test_loss, test_pck, test_oks = run_epoch(model, test_loader, optimizer=None)

print("\n========== FINAL TEST RESULTS ==========")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test PCK : {test_pck:.2f}%")
print(f"Test OKS : {test_oks:.4f}")
print("========================================")

# =========================================================
# 10. PLOTS
# =========================================================
plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"Loss Curve ({MODEL_MODE}, {SPLIT_NAME})")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["train_pck"], label="Train PCK")
plt.plot(history["val_pck"], label="Val PCK")
plt.xlabel("Epoch")
plt.ylabel("PCK (%)")
plt.title(f"PCK Curve ({MODEL_MODE}, {SPLIT_NAME})")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["train_oks"], label="Train OKS")
plt.plot(history["val_oks"], label="Val OKS")
plt.xlabel("Epoch")
plt.ylabel("OKS")
plt.title(f"OKS Curve ({MODEL_MODE}, {SPLIT_NAME})")
plt.legend()
plt.grid(True)
plt.show()

# =========================================================
# 11. VISUALIZATION OF SAMPLE PREDICTIONS
# =========================================================
def draw_pose(ax, keypoints_px, valid_mask, color="r", label=None):
    for c in SKELETON_CONNECTIONS:
        i, j = c
        if valid_mask[i] and valid_mask[j]:
            p1 = keypoints_px[i]
            p2 = keypoints_px[j]
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color=color, linewidth=2)

    pts = keypoints_px[valid_mask]
    if len(pts) > 0:
        ax.scatter(pts[:, 0], pts[:, 1], c=color, s=25, label=label)

model.eval()
num_show = min(3, len(test_dataset))

for idx in range(num_show):
    sample = test_dataset[idx]
    radar_input = sample["radar"].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(radar_input).cpu().numpy()[0]

    gt = sample["keypoints"].numpy()
    valid = sample["valid"].numpy()

    pred_px = pred * np.array([IMG_W, IMG_H], dtype=np.float32)
    gt_px = gt * np.array([IMG_W, IMG_H], dtype=np.float32)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # show radar maps
    if sample["radar"].shape[0] == 2:
        axes[0].imshow(sample["radar"][0].numpy())
        axes[0].set_title("Horizontal Radar")
        axes[1].imshow(sample["radar"][1].numpy())
        axes[1].set_title("Vertical Radar")
    else:
        axes[0].imshow(sample["radar"][0].numpy())
        axes[0].set_title("Horizontal Radar")
        axes[1].axis("off")

    # pose on blank image plane
    canvas = np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8)
    axes[2].imshow(canvas)
    draw_pose(axes[2], gt_px, valid, color="g", label="Ground Truth")
    draw_pose(axes[2], pred_px, valid, color="r", label="Prediction")
    axes[2].set_title(
        f"Pred vs GT\nSegment={sample['segment']} Frame={sample['frame_id']}"
    )
    axes[2].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
For Fusion Model

In [ ]:
import os
import math
import json
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# =========================================================
# 1. CONFIG
# =========================================================
MODEL_MODE = "FUSION"   # "BASELINE" or "FUSION"
DATASET_PATH = "P1"       # change if needed
SPLIT_NAME = "P1S1"       # "P1S1" or "P1S2"
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-3
NUM_WORKERS = 0           # use 0 on mac/notebooks if worker issues happen
SEED = 42
IMG_W, IMG_H = 640, 480

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("====================================")
print("Starting MMVR Pose Training")
print(f"Device      : {DEVICE}")
print(f"Model Mode  : {MODEL_MODE}")
print(f"Dataset Path: {DATASET_PATH}")
print(f"Split       : {SPLIT_NAME}")
print("====================================")

# Camera / radar params from README
R_ext = np.array([
    [0.997214252753733, -0.0607987476962386, -0.0432116463859367],
    [-0.0630518006224704, -0.996610215186671, -0.0528445779985113],
    [0.0398522840384131, -0.0554219384733665, 0.997667381541953]
], dtype=np.float32)

T_ext = np.array([
    [-0.0648588235850897],
    [ 0.302966783204636 ],
    [ 0.0914051241089986]
], dtype=np.float32)

# =========================================================
# 2. REPRODUCIBILITY
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# =========================================================
# 3. HELPERS
# =========================================================
SKELETON_CONNECTIONS = np.array([
    [13, 15], [11, 13], [14, 16], [12, 14], [11, 12],
    [5, 11], [6, 12], [5, 6], [5, 7], [6, 8], [7, 9],
    [8, 10], [1, 2], [0, 1], [0, 2], [1, 3], [2, 4],
    [3, 5], [4, 6]
], dtype=np.int32)

def normalize_heatmap(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    mean = x.mean()
    std = x.std()
    return (x - mean) / (std + 1e-6)

def get_relative_segment_path(root_dir: str, abs_dir: str) -> str:
    rel = os.path.relpath(abs_dir, root_dir)
    return rel.replace("\\", "/")

def flatten_split_entries(split_entries):
    """
    Converts split structure into a set of normalized relative folder paths.
    Usually entries are like:
      ['d1s1/000', 'd1s1/001', ...]
    """
    out = set()

    if isinstance(split_entries, np.ndarray):
        split_entries = split_entries.tolist()

    for item in split_entries:
        if isinstance(item, bytes):
            item = item.decode("utf-8")
        elif not isinstance(item, str):
            item = str(item)

        item = item.replace("\\", "/").strip().strip("/")
        out.add(item)

    return out

def masked_mse_loss(pred, gt, valid_mask):
    """
    pred: (B, 17, 2)
    gt:   (B, 17, 2)
    valid_mask: (B, 17) bool
    """
    mask = valid_mask.unsqueeze(-1).float()
    sq = (pred - gt) ** 2
    loss = (sq * mask).sum() / (mask.sum() + 1e-8)
    return loss

def pck_counts(pred, gt, valid_mask, threshold=0.10):
    """
    threshold is in normalized image coordinates.
    Returns correct_count, total_valid
    """
    distances = torch.norm(pred - gt, dim=2)   # (B,17)
    correct = ((distances < threshold) & valid_mask).sum().item()
    total = valid_mask.sum().item()
    return correct, total

def compute_pck_percent(correct, total):
    if total == 0:
        return 0.0
    return 100.0 * correct / total

def oks_per_instance(pred_xy, gt_xy, valid_mask, bbox_area, sigmas=None):
    """
    pred_xy, gt_xy: (17,2) in pixel coordinates
    valid_mask: (17,) bool
    bbox_area: scalar
    """
    if sigmas is None:
        # COCO-style approximate sigmas for 17 keypoints
        sigmas = np.array([
            0.026, 0.025, 0.025, 0.035, 0.035,
            0.079, 0.079, 0.072, 0.072, 0.062,
            0.062, 0.107, 0.107, 0.087, 0.087,
            0.089, 0.089
        ], dtype=np.float32)

    valid = valid_mask.astype(bool)
    if valid.sum() == 0:
        return 0.0

    d2 = np.sum((pred_xy - gt_xy) ** 2, axis=1)  # (17,)
    denom = 2 * (sigmas ** 2) * (bbox_area + 1e-6)
    oks = np.exp(-d2 / (denom + 1e-9))
    return float(np.mean(oks[valid]))

def estimate_depth_from_radar_boxes(bbox_hori):
    """
    Simple non-learned depth estimate from radar horizontal box.
    bbox_hori shape: (n,4) [x_min, y_min, x_max, y_max]
    x is depth axis in radar coordinate.
    For P1, usually n=1.
    """
    if bbox_hori is None or len(bbox_hori) == 0:
        return None
    depth_center = 0.5 * (bbox_hori[0, 0] + bbox_hori[0, 2])
    return float(depth_center)

# =========================================================
# 4. DATASET
# =========================================================
class RadarPoseDataset(Dataset):
    def __init__(self, root_dir, view="both", allowed_segments=None):
        self.root_dir = root_dir
        self.view = view
        self.allowed_segments = allowed_segments
        self.samples = []

        self._build_index()

    def _build_index(self):
        for root, _, files in os.walk(self.root_dir):
            rel_segment = get_relative_segment_path(self.root_dir, root)

            # Only keep exact segment folders if split is provided
            if self.allowed_segments is not None:
                if rel_segment not in self.allowed_segments:
                    continue

            radar_files = [f for f in files if f.endswith("_radar.npz")]
            for f in sorted(radar_files):
                base = f.replace("_radar.npz", "")
                radar_path = os.path.join(root, f)
                pose_path = os.path.join(root, base + "_pose.npz")
                bbox_path = os.path.join(root, base + "_bbox.npz")
                meta_path = os.path.join(root, base + "_meta.npz")

                if os.path.exists(pose_path) and os.path.exists(bbox_path):
                    self.samples.append({
                        "radar_path": radar_path,
                        "pose_path": pose_path,
                        "bbox_path": bbox_path,
                        "meta_path": meta_path if os.path.exists(meta_path) else None,
                        "segment": rel_segment,
                        "frame_id": base
                    })

        self.samples.sort(key=lambda x: (x["segment"], x["frame_id"]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        radar_data = np.load(sample["radar_path"])
        hori = normalize_heatmap(radar_data["hm_hori"])
        vert = normalize_heatmap(radar_data["hm_vert"])

        if self.view == "both":
            radar_input = np.stack([hori, vert], axis=0)     # (2,256,128)
        elif self.view == "hori":
            radar_input = np.expand_dims(hori, axis=0)       # (1,256,128)
        else:
            raise ValueError("view must be 'both' or 'hori'")

        pose_data = np.load(sample["pose_path"])
        kp_all = pose_data["kp"]   # (n,17,3)

        # P1 is single subject, so we take index 0
        kp = kp_all[0].astype(np.float32)  # (17,3)
        kp_xy = kp[:, :2].copy()
        kp_score = kp[:, 2].copy()

        # visibility / valid mask
        valid = (kp_score > 0) & ~((kp[:, 0] == 0) & (kp[:, 1] == 0))

        # normalize to [0,1]
        kp_norm = kp_xy.copy()
        kp_norm[:, 0] /= IMG_W
        kp_norm[:, 1] /= IMG_H

        bbox_data = np.load(sample["bbox_path"])
        bbox_i = bbox_data["bbox_i"]       # (n,5)
        bbox_hori = bbox_data["bbox_hori"] # (n,4)

        # area for OKS from image-plane bbox
        if len(bbox_i) > 0:
            x1, y1, x2, y2 = bbox_i[0, :4]
            bbox_area = max((x2 - x1) * (y2 - y1), 1.0)
        else:
            bbox_area = 1.0

        # simple depth proxy for later analysis
        depth_target = estimate_depth_from_radar_boxes(bbox_hori)
        if depth_target is None:
            depth_target = 0.0

        global_frame_id = -1
        if sample["meta_path"] is not None:
            meta_data = np.load(sample["meta_path"])
            if "global_frame_id" in meta_data:
                global_frame_id = int(meta_data["global_frame_id"])

        return {
            "radar": torch.tensor(radar_input, dtype=torch.float32),
            "keypoints": torch.tensor(kp_norm, dtype=torch.float32),
            "valid": torch.tensor(valid, dtype=torch.bool),
            "bbox_area": torch.tensor(bbox_area, dtype=torch.float32),
            "depth_target": torch.tensor(depth_target, dtype=torch.float32),
            "segment": sample["segment"],
            "frame_id": sample["frame_id"],
            "global_frame_id": global_frame_id
        }

# =========================================================
# 5. MODEL
# =========================================================
class RadarPoseModel(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),   # 256x128 -> 128x64

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),   # 128x64 -> 64x32

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),   # 64x32 -> 32x16

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),   # 32x16 -> 16x8

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.pose_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 17 * 2)
        )

    def forward(self, radar):
        feat = self.features(radar)
        out = self.pose_head(feat)
        out = out.view(-1, 17, 2)

        # constrain to normalized image plane
        out = torch.sigmoid(out)
        return out

# =========================================================
# 6. SPLIT LOADING
# =========================================================
def load_mmvr_split(dataset_path, split_name="P1S1"):
    split_file = os.path.join(dataset_path, "data_split.npz")
    if not os.path.exists(split_file):
        raise FileNotFoundError(f"Missing split file: {split_file}")

    split_data = np.load(split_file, allow_pickle=True)
    data_split_dict = split_data["data_split_dict"].item()

    if split_name not in data_split_dict:
        raise ValueError(f"Split {split_name} not found in data_split.npz")

    split_info = data_split_dict[split_name]

    train_segments = flatten_split_entries(split_info["train"])
    val_segments = flatten_split_entries(split_info["val"])
    test_segments = flatten_split_entries(split_info["test"])

    return train_segments, val_segments, test_segments

view_setting = "both" if MODEL_MODE == "FUSION" else "hori"
channels = 2 if MODEL_MODE == "FUSION" else 1

train_segments, val_segments, test_segments = load_mmvr_split(DATASET_PATH, SPLIT_NAME)

train_dataset = RadarPoseDataset(DATASET_PATH, view=view_setting, allowed_segments=train_segments)
val_dataset   = RadarPoseDataset(DATASET_PATH, view=view_setting, allowed_segments=val_segments)
test_dataset  = RadarPoseDataset(DATASET_PATH, view=view_setting, allowed_segments=test_segments)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples  : {len(val_dataset)}")
print(f"Test samples : {len(test_dataset)}")

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=False
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False
)

# =========================================================
# 7. TRAIN / EVAL FUNCTIONS
# =========================================================
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_valid = 0

    oks_scores = []

    for batch in loader:
        radar = batch["radar"].to(DEVICE)
        gt = batch["keypoints"].to(DEVICE)
        valid = batch["valid"].to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            pred = model(radar)
            loss = masked_mse_loss(pred, gt, valid)

            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item()

        c, t = pck_counts(pred.detach(), gt.detach(), valid.detach(), threshold=0.10)
        total_correct += c
        total_valid += t

        # OKS on CPU
        pred_np = pred.detach().cpu().numpy()
        gt_np = gt.detach().cpu().numpy()
        valid_np = valid.detach().cpu().numpy()
        bbox_area_np = batch["bbox_area"].cpu().numpy()

        for i in range(len(pred_np)):
            pred_px = pred_np[i] * np.array([IMG_W, IMG_H], dtype=np.float32)
            gt_px = gt_np[i] * np.array([IMG_W, IMG_H], dtype=np.float32)
            oks = oks_per_instance(pred_px, gt_px, valid_np[i], float(bbox_area_np[i]))
            oks_scores.append(oks)

    avg_loss = total_loss / max(len(loader), 1)
    avg_pck = compute_pck_percent(total_correct, total_valid)
    avg_oks = float(np.mean(oks_scores)) if oks_scores else 0.0

    return avg_loss, avg_pck, avg_oks

# =========================================================
# 8. TRAINING
# =========================================================
model = RadarPoseModel(in_channels=channels).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {
    "train_loss": [],
    "train_pck": [],
    "train_oks": [],
    "val_loss": [],
    "val_pck": [],
    "val_oks": []
}

best_val_pck = -1.0
best_model_path = f"best_model_{MODEL_MODE.lower()}_{SPLIT_NAME.lower()}.pth"

print("\nStarting training...\n")
for epoch in range(EPOCHS):
    train_loss, train_pck, train_oks = run_epoch(model, train_loader, optimizer=optimizer)
    val_loss, val_pck, val_oks = run_epoch(model, val_loader, optimizer=None)

    history["train_loss"].append(train_loss)
    history["train_pck"].append(train_pck)
    history["train_oks"].append(train_oks)
    history["val_loss"].append(val_loss)
    history["val_pck"].append(val_pck)
    history["val_oks"].append(val_oks)

    if val_pck > best_val_pck:
        best_val_pck = val_pck
        torch.save(model.state_dict(), best_model_path)

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f}, PCK: {train_pck:.2f}%, OKS: {train_oks:.4f} | "
        f"Val Loss: {val_loss:.4f}, PCK: {val_pck:.2f}%, OKS: {val_oks:.4f}"
    )

print(f"\nBest model saved to: {best_model_path}")

# Save history
history_path = f"history_{MODEL_MODE.lower()}_{SPLIT_NAME.lower()}.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"Training history saved to: {history_path}")

# =========================================================
# 9. TEST EVALUATION
# =========================================================
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
test_loss, test_pck, test_oks = run_epoch(model, test_loader, optimizer=None)

print("\n========== FINAL TEST RESULTS ==========")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test PCK : {test_pck:.2f}%")
print(f"Test OKS : {test_oks:.4f}")
print("========================================")

# =========================================================
# 10. PLOTS
# =========================================================
plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"Loss Curve ({MODEL_MODE}, {SPLIT_NAME})")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["train_pck"], label="Train PCK")
plt.plot(history["val_pck"], label="Val PCK")
plt.xlabel("Epoch")
plt.ylabel("PCK (%)")
plt.title(f"PCK Curve ({MODEL_MODE}, {SPLIT_NAME})")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["train_oks"], label="Train OKS")
plt.plot(history["val_oks"], label="Val OKS")
plt.xlabel("Epoch")
plt.ylabel("OKS")
plt.title(f"OKS Curve ({MODEL_MODE}, {SPLIT_NAME})")
plt.legend()
plt.grid(True)
plt.show()

# =========================================================
# 11. VISUALIZATION OF SAMPLE PREDICTIONS
# =========================================================
def draw_pose(ax, keypoints_px, valid_mask, color="r", label=None):
    for c in SKELETON_CONNECTIONS:
        i, j = c
        if valid_mask[i] and valid_mask[j]:
            p1 = keypoints_px[i]
            p2 = keypoints_px[j]
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color=color, linewidth=2)

    pts = keypoints_px[valid_mask]
    if len(pts) > 0:
        ax.scatter(pts[:, 0], pts[:, 1], c=color, s=25, label=label)

model.eval()
num_show = min(3, len(test_dataset))

for idx in range(num_show):
    sample = test_dataset[idx]
    radar_input = sample["radar"].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(radar_input).cpu().numpy()[0]

    gt = sample["keypoints"].numpy()
    valid = sample["valid"].numpy()

    pred_px = pred * np.array([IMG_W, IMG_H], dtype=np.float32)
    gt_px = gt * np.array([IMG_W, IMG_H], dtype=np.float32)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # show radar maps
    if sample["radar"].shape[0] == 2:
        axes[0].imshow(sample["radar"][0].numpy())
        axes[0].set_title("Horizontal Radar")
        axes[1].imshow(sample["radar"][1].numpy())
        axes[1].set_title("Vertical Radar")
    else:
        axes[0].imshow(sample["radar"][0].numpy())
        axes[0].set_title("Horizontal Radar")
        axes[1].axis("off")

    # pose on blank image plane
    canvas = np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8)
    axes[2].imshow(canvas)
    draw_pose(axes[2], gt_px, valid, color="g", label="Ground Truth")
    draw_pose(axes[2], pred_px, valid, color="r", label="Prediction")
    axes[2].set_title(
        f"Pred vs GT\nSegment={sample['segment']} Frame={sample['frame_id']}"
    )
    axes[2].legend()

    plt.tight_layout()
    plt.show()